<a href="https://colab.research.google.com/github/GGSimmons1992/UTYV6k8pXAuLL0yL/blob/main/Notebooks/Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1b: Classical Random Forest Training

In [1]:
!pip install category_encoders

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier as rf
import pickle
from imblearn.over_sampling import SMOTENC
from os.path import exists
from sklearn.preprocessing import StandardScaler
import category_encoders as ce
from sklearn.feature_selection import chi2
from scipy.stats import spearmanr
import json
from sklearn.metrics import f1_score
from google.colab import drive

drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Src/')
import dataPrep

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def splitBetweenTrainAndDev(df):
  train,dev = train_test_split(df,test_size=0.2,random_state=51)
  return train,dev

In [4]:
def trainAndTestModel(model, train, dev):
  model.fit(train.drop(columns=['isSubscribed']), train['isSubscribed'])
  train_predictions = model.predict(train.drop(columns=['isSubscribed']))
  dev_predictions = model.predict(dev.drop(columns=['isSubscribed']))
  trainScore = f1_score(train['isSubscribed'], train_predictions)
  testScore = f1_score(dev['isSubscribed'], dev_predictions)
  return trainScore, testScore

In [5]:
def createModel(criterion, nEstimators, maxDepth, maxFeatures):
  return rf(criterion=criterion, n_estimators=nEstimators, max_depth=maxDepth, max_features=maxFeatures)

In [6]:
def createSortedScoreVsHyperparameterDF(scoreVsHyperparameterDictionary):
  return pd.DataFrame(scoreVsHyperparameterDictionary).sort_values(by=['devScore','trainScore'],ascending=False)

In [7]:
def appendToHyperparameterDictionary(scoreVsHyperparameterDictionary, criterion, nEstimators, maxDepth, maxFeatures, trainScore, devScore):
  scoreVsHyperparameterDictionary['criterion'].append(criterion)
  scoreVsHyperparameterDictionary['nEstimators'].append(nEstimators)
  scoreVsHyperparameterDictionary['maxDepth'].append(maxDepth)
  scoreVsHyperparameterDictionary['maxFeatures'].append(maxFeatures)
  scoreVsHyperparameterDictionary['trainScore'].append(trainScore)
  scoreVsHyperparameterDictionary['devScore'].append(devScore)
  return scoreVsHyperparameterDictionary

In [8]:
def modelExistsInDrive(filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Models/{filename}"
  return exists(fullName)

In [9]:
def saveModelToDrive(data,filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Models/{filename}"
  pickle.dump(data, open(fullName, 'wb'))

In [10]:
def retrieveModelFromDrive(filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Models/{filename}"
  return pickle.load(open(fullName, 'rb'))

In [11]:
def adjustRange(dfColumn):
  # Ensure min and max are called as methods on a pandas Series
  col_min = dfColumn.min()
  col_max = dfColumn.max()

  if col_min != col_max:
    return np.arange(col_min, col_max + 1, 1) # Include col_max in the range

  # Handle the case where all values are the same
  return np.arange(2, 2 * col_max, 1)




In [12]:
def main():
  train = dataPrep.retrieveCSVFromDrive("SalesReinforcerTrain.csv")
  dev = dataPrep.retrieveCSVFromDrive("SalesReinforcerDev.csv")
  nColumns = len(train.columns)
  bestDevScore = 0

  possibleCriterion = ["gini","entropy","log_loss"]
  possibleNEstimators = np.arange(5,15, 1)
  possibleMaxDepth = np.arange(2,14,1)
  possibleMaxFeatures = np.arange(2,8,1)

  scoreVsHyperparameterDictionary = {
      "criterion":[],
      "nEstimators":[],
      "maxDepth":[],
      "maxFeatures":[],
      "trainScore":[],
      "devScore":[]
  }
  maxIterations = 2100

  for iteration in range(maxIterations):
    print(f"iteration: {iteration+1} of {maxIterations}")
    criterion = np.random.choice(possibleCriterion)
    nEstimators = np.random.choice(possibleNEstimators)
    maxDepth = np.random.choice(possibleMaxDepth)
    maxFeatures = np.random.choice(possibleMaxFeatures)

    model = createModel(criterion, nEstimators, maxDepth, maxFeatures)
    trainScore, devScore = trainAndTestModel(model, train, dev)

    # 1. Unconditionally track and save the overall best model to prevent losing progress
    if (devScore >= bestDevScore):
      bestDevScore = devScore
      saveModelToDrive(model, "baseRandomForest.pkl")
      print(f"-> New best model saved with dev score: {devScore:.4f} (train: {trainScore:.4f})")

      if (trainScore >= .5) and (devScore >= .5):
        print("🎉 Target threshold of 0.5 reached!")

    scoreVsHyperparameterDictionary = appendToHyperparameterDictionary(scoreVsHyperparameterDictionary,
                                                                      criterion, nEstimators, maxDepth,
                                                                      maxFeatures, trainScore, devScore)
    if (iteration == (maxIterations-1)):
      df = createSortedScoreVsHyperparameterDF(scoreVsHyperparameterDictionary)
      display(df)
      dataPrep.saveCSVToDrive(df,"BestHyperparameters.csv")
      if (modelExistsInDrive("baseRandomForest.pkl")):
        print('best model found')
      else:
        print('best model not found')
      return

    if (iteration % 10) == 9:
      bestFiveHyperparameterGroups = createSortedScoreVsHyperparameterDF(scoreVsHyperparameterDictionary).head(5)
      display(bestFiveHyperparameterGroups)
      if (bestFiveHyperparameterGroups.iloc[0].devScore >= bestDevScore):
        possibleNEstimators = adjustRange(bestFiveHyperparameterGroups['nEstimators'])
        possibleMaxDepth = adjustRange(bestFiveHyperparameterGroups['maxDepth'])
        possibleMaxFeatures = adjustRange(bestFiveHyperparameterGroups['maxFeatures'])
      if (modelExistsInDrive("baseRandomForest.pkl")):
        devSet = dataPrep.retrieveCSVFromDrive("SalesReinforcerDev.csv")
        bestModel = retrieveModelFromDrive("baseRandomForest.pkl")
        dev_predictions = bestModel.predict(devSet.drop(columns=['isSubscribed']))
        current_f1 = f1_score(devSet['isSubscribed'], dev_predictions)

        # Early exit only if we hit the desired high-performance threshold of 0.75
        if current_f1 >= 0.75:
          print(f"Target F1 met! Final dev score: {current_f1}")
          return

In [13]:
if __name__ == "__main__":
  main()

iteration: 1 of 2100
-> New best model saved with dev score: 0.0000 (train: 0.9996)
iteration: 2 of 2100
-> New best model saved with dev score: 0.0000 (train: 0.9755)
iteration: 3 of 2100
-> New best model saved with dev score: 0.0000 (train: 0.9789)
iteration: 4 of 2100
-> New best model saved with dev score: 0.0000 (train: 0.9990)
iteration: 5 of 2100
-> New best model saved with dev score: 0.3333 (train: 0.9995)
iteration: 6 of 2100
iteration: 7 of 2100
iteration: 8 of 2100
iteration: 9 of 2100
iteration: 10 of 2100


,criterion,nEstimators,maxDepth,maxFeatures,trainScore,devScore
4,gini,10,11,2,0.999496,0.333333
8,gini,7,2,2,0.884653,0.181818
6,entropy,10,16,3,0.999748,0.000000
9,log_loss,7,17,3,0.999622,0.000000
0,gini,7,12,3,0.999622,0.000000


iteration: 11 of 2100
iteration: 12 of 2100
iteration: 13 of 2100
iteration: 14 of 2100
iteration: 15 of 2100
iteration: 16 of 2100
iteration: 17 of 2100
iteration: 18 of 2100
iteration: 19 of 2100
iteration: 20 of 2100


,criterion,nEstimators,maxDepth,maxFeatures,trainScore,devScore
4,gini,10,11,2,0.999496,0.333333
8,gini,7,2,2,0.884653,0.181818
15,gini,9,2,3,0.879109,0.068966
6,entropy,10,16,3,0.999748,0.000000
14,entropy,9,13,3,0.999748,0.000000


iteration: 21 of 2100
iteration: 22 of 2100
iteration: 23 of 2100
iteration: 24 of 2100
iteration: 25 of 2100
iteration: 26 of 2100
iteration: 27 of 2100
iteration: 28 of 2100
iteration: 29 of 2100
iteration: 30 of 2100


,criterion,nEstimators,maxDepth,maxFeatures,trainScore,devScore
4,gini,10,11,2,0.999496,0.333333
8,gini,7,2,2,0.884653,0.181818
28,entropy,8,4,3,0.958518,0.117647
15,gini,9,2,3,0.879109,0.068966
6,entropy,10,16,3,0.999748,0.000000


... (411 outputs left out)

,criterion,nEstimators,maxDepth,maxFeatures,trainScore,devScore
163,entropy,8,11,3,0.999874,0.5
523,entropy,3,11,3,0.999748,0.4
1573,gini,6,11,4,0.999748,0.4
795,log_loss,6,10,3,0.999748,0.4
1438,entropy,6,10,5,0.999622,0.4


iteration: 2091 of 2100
iteration: 2092 of 2100
iteration: 2093 of 2100
iteration: 2094 of 2100
iteration: 2095 of 2100
iteration: 2096 of 2100
iteration: 2097 of 2100
iteration: 2098 of 2100
iteration: 2099 of 2100
iteration: 2100 of 2100


,criterion,nEstimators,maxDepth,maxFeatures,trainScore,devScore
163,entropy,8,11,3,0.999874,0.5
523,entropy,3,11,3,0.999748,0.4
1573,gini,6,11,4,0.999748,0.4
795,log_loss,6,10,3,0.999748,0.4
1438,entropy,6,10,5,0.999622,0.4
...,...,...,...,...,...,...
52,entropy,9,3,3,0.882180,0.0
140,entropy,9,4,2,0.881934,0.0
154,log_loss,8,4,2,0.881811,0.0
45,gini,10,4,3,0.881564,0.0


best model found
